In [17]:
%%capture
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install emoji
!pip install "accelerate>=0.26.0"

In [18]:
import os
import re
import shutil

import emoji
import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [19]:
def clean_text(text):
    if pd.isna(text):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. remove @USER mentions
    text = re.sub(r"@user", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@url", "", text, flags=re.IGNORECASE)

    # 3. remove URLs (actual links or placeholder "URL")
    # text = re.sub(r'http\S+|https\S+|url', '', text, flags=re.IGNORECASE)

    # 4. remove underscores, repeated underscores
    text = re.sub(r"_+", " ", text)

    # 5. remove slashes
    text = text.replace("\\", " ").replace("/", " ")

    # 6. remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 7. remove quotation marks (normal + smart)
    text = re.sub(r"[\"“”]", "", text)

    # 8. normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [20]:
# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [21]:
%%time
# Load Model & Tokenizer
MODEL_NAME = "FacebookAI/xlm-roberta-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

base_model.to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 4.49 s, sys: 2.88 s, total: 7.37 s
Wall time: 7.01 s


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, ou

In [22]:
# Load Data
DATA_DIR = "data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")


def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df = df[["text", "polarization"]]
            df["lang"] = lang
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

Loading Train Data...
Loaded 73681 training examples
Loading Dev Data (Used as internal Test)...
Loaded 3687 dev examples
Loading Test Data (For Submission)...
Loaded 33288 test examples


In [23]:
# Preprocess Text
print("Preprocessing text (cleaning)...")
raw_train_df["text"] = raw_train_df["text"].astype(str).apply(clean_text)
raw_dev_df["text"] = raw_dev_df["text"].astype(str).apply(clean_text)
raw_test_df["text"] = raw_test_df["text"].astype(str).apply(clean_text)

Preprocessing text (cleaning)...


In [24]:
# Data Splitting
# Rename 'polarization' to 'labels'
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_test_df.columns:
    raw_test_df = raw_test_df.rename(columns={"polarization": "labels"})

In [25]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(raw_train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(raw_dev_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(raw_test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict(
    {"train": train_dataset, "validation": val_dataset, "test": test_dataset}
)

In [26]:
pd.DataFrame(train_dataset[:5]).head()

,text,labels
0,ఒత్తిడిని ఒంటరిగా భరించాల్సిన అవసరం లేదు. ఎవరై...,0
1,సైనికుల కుటుంబాలు వారు విధులలో ఉన్నప్పుడు అపార...,0
2,ఒక వ్యక్తి యొక్క లింగ గుర్తింపును గౌరవించకుండా...,1
3,వివిధ దేశాల సంస్కృతుల గురించి తెలుసుకోవడం ద్వా...,1
4,రాజకీయ రంగంలో ప్రజాస్వామ్య విలువలను కాపాడే బాధ...,0


In [27]:
%%time


# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


print("Tokenizing datasets...")
encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

Tokenizing datasets...


Map:   0%|          | 0/73681 [00:00<?, ? examples/s]

Map:   0%|          | 0/3687 [00:00<?, ? examples/s]

Map:   0%|          | 0/33288 [00:00<?, ? examples/s]

CPU times: user 16.8 s, sys: 320 ms, total: 17.1 s
Wall time: 8 s


In [28]:
# Training Arguments
OUTPUT_DIR = "./output_results"
BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-5
GRAD_ACCUM = 2

# Calculate steps
steps_per_epoch = len(encoded_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
eval_steps = steps_per_epoch
steps_per_epoch

1151

In [29]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 60,  # Save less frequently to save space
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    eval_strategy="steps",
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")[
        "f1"
    ]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}


trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [30]:
%%time
# Start Training
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy
1151,0.535500,0.441887,0.797354,0.797667
2302,0.419600,0.389883,0.821226,0.821806
3453,0.343700,0.408821,0.827880,0.828858
4604,0.269200,0.430071,0.822864,0.822891
5755,0.209400,0.504044,0.815919,0.817196
6906,0.160200,0.576166,0.819197,0.820450


CPU times: user 37min 11s, sys: 8min 50s, total: 46min 2s
Wall time: 46min 9s


TrainOutput(global_step=6906, training_loss=0.32293832195070976, metrics={'train_runtime': 2768.987, 'train_samples_per_second': 1330.469, 'train_steps_per_second': 20.802, 'total_flos': 2.0583987390939034e+17, 'train_loss': 0.32293832195070976, 'epoch': 5.995223621363439})

In [31]:
%%time
# Evaluation on Internal Test Set (Dev Folder)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
report = classification_report(
    true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4
)
print(f"\n{report}")

macro_f1 = f1_score(true_labels, pred_labels, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

# Per-Language Analysis
raw_test_df["preds"] = pred_labels
print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(raw_test_df["lang"].unique()):
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append(
        {"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)}
    )

results_df = pd.DataFrame(results)
print(f"\nAverage Macro F1 across languages: {results_df['f1_macro'].mean():.4f}")

Evaluating on Internal Test Set (Dev folder data)...



Classification Report:

               precision    recall  f1-score   support

Not Polar (0)     0.8146    0.7995    0.8070     15562
    Polar (1)     0.8268    0.8402    0.8335     17726

     accuracy                         0.8212     33288
    macro avg     0.8207    0.8199    0.8202     33288
 weighted avg     0.8211    0.8212    0.8211     33288

Macro F1: 0.8202

=== Macro F1 per Language ===
amh: F1=0.7603, Acc=0.8208, Support=1501
arb: F1=0.8083, Acc=0.8100, Support=1521
ben: F1=0.8251, Acc=0.8274, Support=1501
deu: F1=0.7030, Acc=0.7032, Support=1432
eng: F1=0.7784, Acc=0.7955, Support=1452
fas: F1=0.7878, Acc=0.8423, Support=1484
hau: F1=0.6867, Acc=0.8856, Support=1644
hin: F1=0.7680, Acc=0.8932, Support=1236
ita: F1=0.6463, Acc=0.6678, Support=1538
khm: F1=0.7287, Acc=0.9234, Support=2988
mya: F1=0.8718, Acc=0.8755, Support=1301
nep: F1=0.8892, Acc=0.8893, Support=903
ori: F1=0.7712, Acc=0.8246, Support=1066
pan: F1=0.7688, Acc=0.7689, Support=809
pol: F1=0.7777, Acc=0.